In [ ]:
# Importation de la session Snowpark
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# On force l'utilisation de la base de données et du schéma
session.use_database("HOUSE_PRICE_DB")
session.use_schema("RAW_DATA")

print(f"Connecté à : {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
import snowflake.snowpark.functions as F

# On utilise le nom complet pour être sûr de trouver la table
df = session.table("HOUSE_PRICE_DB.RAW_DATA.HOUSE_DATA")

# 1. Affichage des premières lignes pour vérifier
print("--- Aperçu des données ---")
df.show(5)

# 2. Statistiques descriptives
print("--- Statistiques descriptives ---")
df.describe().show()

# 3. Vérification des valeurs nulles
print("--- Valeurs manquantes ---")
df.select([F.count(F.when(F.col(c).is_null(), c)).alias(c) for c in df.columns]).show()

In [ ]:
# --- Cell 3: Data Visualization (EDA) ---
import matplotlib.pyplot as plt
import seaborn as sns

# Convert Snowpark DataFrame to Pandas for visualization
# Note: Snowflake processing is done, now we use Pandas for plotting
pdf = df.to_pandas()

# 1. Plot: Price Distribution (Target Variable Analysis)
plt.figure(figsize=(10, 5))
sns.histplot(pdf['PRICE'], kde=True, color='skyblue')
plt.title('House Price Distribution (Target Variable)')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

# 2. Plot: Correlation Matrix (Identifying feature relationships)
# This fulfills the "Comprendre la corrélation entre les colonnes" requirement
plt.figure(figsize=(10, 8))
numeric_pdf = pdf.select_dtypes(include=['number'])
sns.heatmap(numeric_pdf.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix (EDA)')
plt.show()

# 3. Plot: Price vs Surface Area (Relationship Analysis)
plt.figure(figsize=(10, 5))
sns.scatterplot(data=pdf, x='AREA', y='PRICE', hue='AIRCONDITIONING')
plt.title('Price vs Area (Influence of Air Conditioning)')
plt.xlabel('Surface Area')
plt.ylabel('Price')
plt.show()